# 15 - Tool Calling with LangChain

This notebook moves from raw tool calling to the higher-level LangChain patterns.

You will learn:
- `@tool`
- `bind_tools`
- iterative tool loops
- `create_agent`
- error handling
- model/tool configuration

The current LangChain documentation recommends `create_agent()` as the standard agent entry point, while lower-level `bind_tools()` is useful for learning and custom control.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langchain langchain-openai python-dotenv

## 1. Define multiple tools

Tool descriptions are part of the model's decision context. A vague tool description can produce incorrect tool choices.

In [ ]:
from langchain.tools import tool

@tool
def add(a: float, b: float) -> float:
    'Add two numbers.'
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    'Multiply two numbers.'
    return a * b

tools = [add, multiply]
for t in tools:
    print(t.name, '=>', t.description)

## 2. Low-level loop

The explicit loop is excellent for learning because nothing is hidden.

```text
model
 ↓
tool calls?
 ↓ yes
execute
 ↓
send tool result
 ↓
model again
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

if CHAT_MODEL:
    model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    model_with_tools = model.bind_tools(tools)

    messages = [HumanMessage(content='What is 12 * 8?')]
    for step in range(5):
        ai = model_with_tools.invoke(messages)
        messages.append(ai)

        if not ai.tool_calls:
            print(ai.content)
            break

        for call in ai.tool_calls:
            selected = {t.name: t for t in tools}[call['name']]
            result = selected.invoke(call['args'])
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call['id'],
                )
            )
else:
    print('Configure OPENAI_CHAT_MODEL first.')

## 3. Current agent abstraction

For standard agent behavior, LangChain provides `create_agent`.

This hides the repetitive loop while still exposing the agent state/result.

In [ ]:
from langchain.agents import create_agent

if CHAT_MODEL:
    agent = create_agent(
        model=CHAT_MODEL,
        tools=tools,
        system_prompt='You are a careful arithmetic assistant.',
    )

    # result = agent.invoke({
    #     'messages': [{'role': 'user', 'content': 'What is 12 * 8?'}]
    # })
    # print(result['messages'][-1].content)
else:
    print('Configure OPENAI_CHAT_MODEL first.')

## 4. Tool error handling

Tools fail in real systems.

Handle:
- timeouts
- validation errors
- rate limits
- permissions
- unavailable services
- malformed responses

Return a controlled error rather than crashing the whole agent.

In [ ]:
def safe_tool_call(tool, args):
    try:
        return {'ok': True, 'result': tool.invoke(args)}
    except Exception as exc:
        return {'ok': False, 'error': str(exc)}

print(safe_tool_call(add, {'a': 2, 'b': 5}))

## 5. When to use what

Use `bind_tools` when learning or when you need custom message/tool-loop control.

Use `create_agent` for a standard tool-using agent.

Move to LangGraph when you need explicit state transitions, branching, persistence, interrupts, or more complex workflows.

## 6. Practice

1. Add division.
2. Create a deliberate tool failure.
3. Compare the explicit loop with `create_agent`.
4. Measure number of model calls.

## Evaluation

Evaluate:
- tool selection
- argument correctness
- task completion
- unnecessary tool calls
- failure recovery
- latency

Improvement:
- clearer tool descriptions
- stricter schemas
- tool budgets
- retries with limits
- move complex workflows to LangGraph.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/